In [33]:
# ===== 1: 라이브러리, 데이터 로드 =====
import re
from collections import Counter

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import sparse
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

train = pd.read_csv("./sample/train.csv")
test = pd.read_csv("./sample/test.csv")

gene_cols = list(train.columns[2:])                      # ID, SUBCLASS 제외
assert set(gene_cols) <= set(test.columns), "test 컬럼이 train과 다름"

print("train 결측 칸:", int(train[gene_cols].isnull().sum().sum()),
    "| test 결측 칸:", int(test[gene_cols].isnull().sum().sum()),
    f"(환자 {int(test[gene_cols].isnull().any(axis=1).sum())}명)")

train 결측 칸: 0 | test 결측 칸: 237 (환자 128명)


In [34]:
# ===== 2: 변이 표기를 0~3 점수로 변환 =====
# 0 = WT, 1 = synonymous(같은 아미노산), 2 = missense/in-frame, 3 = 절단(nonsense, frameshift)
NO_CHANGE = re.compile(r'[A-Z]\d+')                      # 위치만 있고 변화 표기가 없는 형태 (예: M364)


def encode_token(t: str) -> int:
    """변이 하나를 0~3 점수로 변환. 해석 못하면 -1"""
    if 'fs' in t:                                        # frameshift: L1854fs
        return 3
    if '>' in t:                                         # 355_356QG>H*, 91_92NH>KY
        left, right = t.split('>')
        left = re.sub(r'^[\d_\-]+', '', left)
        if right.endswith('*'):
            return 3
        return 1 if left == right else 2
    if re.fullmatch(r'[A-Z]\d+\*', t):                   # nonsense: E237*
        return 3
    if re.fullmatch(r'\*\d+\*', t):                      # 종결코돈 유지: *261*
        return 1
    if 'del' in t or 'ins' in t:                         # in-frame 결실/삽입: R649del, G103_K104insFF
        return 3 if t.endswith('*') else 2
    m = re.fullmatch(r'([A-Z])\d+([A-Z])', t)            # 단일 아미노산 치환
    if m:
        return 1 if m.group(1) == m.group(2) else 2
    if NO_CHANGE.fullmatch(t):                           # 위치만 있는 변이(M364): 변이는 있으므로 2점으로 가정
        return 2
    return -1


def encode_cell(v: str) -> int:
    if v == 'WT':
        return 0
    return max(encode_token(t) for t in v.split())       # 한 칸에 변이가 여러 개면 최댓값


def unknown_tokens(values):
    """칸 안의 일부 변이만 해석 못해도 잡아내기 위해 토큰 단위로 확인"""
    return sorted({t for v in values if v != 'WT' for t in v.split() if encode_token(t) == -1})


def encode_frame(df_raw):
    """문자열 변이 표 -> int8 점수 표. 고유 문자열만 변환해서 빠르게 처리"""
    df_raw = df_raw.fillna('WT')                         # 결측은 변이 없음(WT)으로
    uniq = pd.unique(df_raw.values.ravel())
    bad = unknown_tokens(uniq)
    assert not bad, f"해석 못한 변이 표기: {bad[:20]}"

    fb_vals = [v for v in uniq if v != 'WT' and any(NO_CHANGE.fullmatch(t) for t in v.split())]
    if fb_vals:
        fb = sorted({t for v in fb_vals for t in v.split() if NO_CHANGE.fullmatch(t)})
        print(f"주의: 변화 표기 없이 위치만 있는 변이 {fb} -> 2점으로 처리 "
            f"({int(df_raw.isin(fb_vals).sum().sum())}칸)")

    mp = {v: encode_cell(v) for v in uniq}
    return df_raw.apply(lambda col: col.map(mp)).astype(np.int8), df_raw


X, raw_train = encode_frame(train[gene_cols])
X_test, raw_test = encode_frame(test[gene_cols])

y = train['SUBCLASS']
le = LabelEncoder()
y_enc = le.fit_transform(y)                              # 26개 암종 -> 0~25
n_class = len(le.classes_)
print(X.shape, X_test.shape, "클래스 수:", n_class)


주의: 변화 표기 없이 위치만 있는 변이 ['M364', 'M391'] -> 2점으로 처리 (1칸)
(6201, 4384) (2546, 4384) 클래스 수: 26


In [35]:
# ===== 3: 특징 만들기 (유전자 점수 + 변이 개수 + 반복되는 변이) =====
MIN_COUNT = 3        # 유전자: 학습 환자 중 변이가 있는 환자가 이 수 이상일 때만 사용
MIN_VAR = 5          # 반복되는 변이: 학습 환자 중 이 수 이상에서 나온 것만 사용
USE_TMB = True       # False로 바꾸면 TMB, n_syn, n_mis, n_trunc 제외


def extract_variants(raw, skip_syn=True):
    """(환자 행번호, '유전자|변이') 쌍 목록. WT와 침묵 변이는 제외"""
    vals, genes = raw.values, raw.columns.values
    r, c = np.nonzero(vals != 'WT')
    pairs = []
    for i, j in zip(r, c):
        for t in vals[i, j].split():                     # 한 칸에 변이가 여러 개여도 각각 처리
            if skip_syn and re.fullmatch(r'([A-Z])\d+\1', t):
                continue
            pairs.append((i, f"{genes[j]}|{t}"))
    return pairs


def to_sparse(pairs, n_rows, vocab):
    rows, cols = [], []
    for i, f in pairs:
        k = vocab.get(f)
        if k is not None:
            rows.append(i)
            cols.append(k)
    M = sparse.csr_matrix(
        (np.ones(len(rows), dtype=np.float32), (rows, cols)),
        shape=(n_rows, len(vocab)),
    )
    M.data[:] = 1
    return M


def dense_features(Xd, keep):
    v = Xd.values
    parts = [v[:, keep]]
    if USE_TMB:
        parts += [(v > 0).sum(1, keepdims=True),         # TMB: 환자 한 명의 변이 유전자 수
                (v == 1).sum(1, keepdims=True),        # n_syn
                (v == 2).sum(1, keepdims=True),        # n_mis
                (v == 3).sum(1, keepdims=True)]        # n_trunc
    return np.hstack(parts).astype(np.float32)


def make_features(Xt, raw_t, Xv, raw_v):
    """필터와 어휘는 Xt(학습 쪽)로만 만들고 Xv에는 적용만 한다 -> 데이터 누수 방지"""
    keep = ((Xt > 0).sum(axis=0) >= MIN_COUNT).values
    pairs_t, pairs_v = extract_variants(raw_t), extract_variants(raw_v)
    cnt = Counter(f for _, f in pairs_t)
    var_names = [f for f, n in cnt.items() if n >= MIN_VAR]
    vocab = {f: k for k, f in enumerate(var_names)}
    Vt = to_sparse(pairs_t, len(raw_t), vocab)
    Vv = to_sparse(pairs_v, len(raw_v), vocab)

    names = list(np.array(Xt.columns)[keep])
    if USE_TMB:
        names += ['TMB', 'n_syn', 'n_mis', 'n_trunc']
    names += var_names

    A = sparse.hstack([sparse.csr_matrix(dense_features(Xt, keep)), Vt]).tocsr()
    B = sparse.hstack([sparse.csr_matrix(dense_features(Xv, keep)), Vv]).tocsr()
    return A, B, names

In [36]:
# ===== 4: 5-fold 교차검증 (OOF 예측) =====
CW = 'balanced'          # 클래스 불균형 보정. 아래 가중치 w도 이 설정으로 튜닝됨
N_EST_CV = 200
N_EST_FINAL = 220
PARAMS = dict(
    objective='multiclass', learning_rate=0.05, num_leaves=15,
    min_child_samples=5, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
    reg_lambda=1.0, class_weight=CW, n_jobs=-1, verbose=-1,
)

skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof = np.zeros((len(y_enc), n_class))
for k, (tr, va) in enumerate(skf.split(X, y_enc)):
    Xt, Xv, _ = make_features(X.iloc[tr], raw_train.iloc[tr], X.iloc[va], raw_train.iloc[va])
    m = lgb.LGBMClassifier(n_estimators=N_EST_CV, random_state=42, **PARAMS)
    m.fit(Xt, y_enc[tr])                                 # 조기종료 없이 고정 -> OOF 점수가 정직함
    oof[va] = m.predict_proba(Xv)
    print(f"fold {k}: Macro F1 {f1_score(y_enc[va], oof[va].argmax(1), average='macro'):.4f}")

print("OOF Macro F1:", round(f1_score(y_enc, oof.argmax(1), average='macro'), 4))


fold 0: Macro F1 0.4100
fold 1: Macro F1 0.4153
fold 2: Macro F1 0.4222
fold 3: Macro F1 0.3970
fold 4: Macro F1 0.4240
OOF Macro F1: 0.4145


In [37]:
# ===== 5: 암종별 확률 가중치 튜닝 (Macro F1 최대화) =====
def tune_class_weights(P, y, n_iter=3):
    grid = np.exp(np.linspace(-1.5, 1.5, 13))
    w = np.ones(P.shape[1])
    best = f1_score(y, (P * w).argmax(1), average='macro')
    for _ in range(n_iter):
        for c in range(P.shape[1]):
            for g in grid:
                w2 = w.copy()
                w2[c] = g
                s = f1_score(y, (P * w2).argmax(1), average='macro')
                if s > best:
                    best, w = s, w2
    return w, best


# 과적합 확인: 절반으로 튜닝하고, 쓰지 않은 나머지 절반에서 평가
rng = np.random.RandomState(0)
idx = rng.permutation(len(y_enc))
a, b = idx[:len(idx) // 2], idx[len(idx) // 2:]
w_a, _ = tune_class_weights(oof[a], y_enc[a])
print("절반 검증  튜닝 전:", round(f1_score(y_enc[b], oof[b].argmax(1), average='macro'), 4),
      " 튜닝 후:", round(f1_score(y_enc[b], (oof[b] * w_a).argmax(1), average='macro'), 4))

# 최종 가중치는 전체 OOF로 튜닝
w, best = tune_class_weights(oof, y_enc)
print("가중치 적용 OOF Macro F1 (튜닝에 쓴 데이터라 낙관적):", round(best, 4))
print(classification_report(y_enc, (oof * w).argmax(1), target_names=le.classes_, zero_division=0))

절반 검증  튜닝 전: 0.4194  튜닝 후: 0.4163
가중치 적용 OOF Macro F1 (튜닝에 쓴 데이터라 낙관적): 0.4364
              precision    recall  f1-score   support

         ACC       0.95      0.82      0.88        72
        BLCA       0.65      0.38      0.48       104
        BRCA       0.63      0.46      0.53       786
        CESC       0.14      0.22      0.17       155
        COAD       0.82      0.70      0.75       223
        DLBC       0.63      0.50      0.56        38
      GBMLGG       0.57      0.21      0.30       461
        HNSC       0.34      0.30      0.32       223
       KIPAN       0.21      0.13      0.16       515
        KIRC       0.23      0.41      0.29       334
        LAML       0.63      0.51      0.57       158
         LGG       0.43      0.77      0.55       229
        LIHC       0.47      0.28      0.35       158
        LUAD       0.30      0.37      0.33       184
        LUSC       0.42      0.26      0.32       178
          OV       0.28      0.58      0.37       253
  

In [38]:
# ===== 6: 전체 train으로 학습 -> test 예측 -> 제출 파일 =====
X_full, X_te, feat_names = make_features(X, raw_train, X_test, raw_test)
print(X_full.shape, X_te.shape)                          # 열 수가 같아야 함

P_test = np.zeros((X_te.shape[0], n_class))
seeds = [42, 43, 44]                                     # seed 3개 평균으로 안정화
for s in seeds:
    m = lgb.LGBMClassifier(n_estimators=N_EST_FINAL, random_state=s, **PARAMS)
    m.fit(X_full, y_enc)
    P_test += m.predict_proba(X_te) / len(seeds)

pred_test = (P_test * w).argmax(1)
labels = le.inverse_transform(pred_test)

# 점검: 예측 비율을 train 비율과 비교 (test 구성이 달라서 어느 정도 차이는 정상)
check = pd.DataFrame({
    'train(%)': y.value_counts(normalize=True) * 100,
    'test예측(%)': pd.Series(labels).value_counts(normalize=True) * 100,
}).round(1).fillna(0)
check['배수'] = (check['test예측(%)'] / check['train(%)']).round(2)
print(check.sort_values('배수'))

# 중요한 특징 상위 20 (gain 기준)
imp = pd.Series(m.booster_.feature_importance(importance_type='gain'),
                index=feat_names).sort_values(ascending=False)
print(imp.head(20).round(0))

submission = pd.DataFrame({'ID': test['ID'], 'SUBCLASS': labels})
submission.to_csv('submission_3.csv', index=False)
print(submission.head(), submission.shape)

# 나중에 다시 쓸 수 있게 중간 결과 저장 (커널을 재시작해도 유지됨)
np.save('oof.npy', oof)
np.save('P_test.npy', P_test)
np.save('w.npy', w)

(6201, 4607) (2546, 4607)
        train(%)  test예측(%)    배수
GBMLGG       7.4        1.5  0.20
LAML         2.5        1.1  0.44
KIPAN        8.3        3.9  0.47
ACC          1.2        0.6  0.50
BRCA        12.7        6.6  0.52
PAAD         1.9        1.0  0.53
THCA         5.2        2.8  0.54
SKCM         4.5        2.7  0.60
HNSC         3.6        2.8  0.78
PRAD         4.3        3.5  0.81
LIHC         2.5        2.1  0.84
PCPG         2.4        2.1  0.88
TGCT         2.0        1.9  0.95
SARC         3.2        3.2  1.00
BLCA         1.7        1.8  1.06
LUSC         2.9        3.2  1.10
LGG          3.7        4.2  1.14
KIRC         5.4        6.2  1.15
UCEC         3.2        3.9  1.22
THYM         1.6        2.1  1.31
OV           4.1        5.9  1.44
LUAD         3.0        4.7  1.57
DLBC         0.6        1.0  1.67
COAD         3.6        7.2  2.00
CESC         2.5        5.7  2.28
STES         6.1       18.3  3.00
TMB           26367.0
n_mis         21475.0
n_syn       